In [0]:
from pyspark.sql import functions as F

CATALOG = "insurance_lakehouse"
QUARANTINE_SCHEMA = "quarantine"

In [0]:
def write_quarantine(df, table_name):
    (
        df.select(
            "record_id",
            "source_table",
            "error_reason",
            "error_severity",
            "quarantine_timestamp",
            "source_file_name",
            "ingest_run_id",
            "original_record_json"
        )
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{QUARANTINE_SCHEMA}.{table_name}")
    )

In [0]:
customers_invalid = spark.table(f"{CATALOG}.silver.silver_customers") \
    .filter(F.col("customer_id").isNull())

policies_invalid = spark.table(f"{CATALOG}.silver.silver_policies") \
    .filter(F.col("policy_id").isNull())

claims_invalid = spark.table(f"{CATALOG}.silver.silver_claims") \
    .filter(F.col("claim_id").isNull())

payments_invalid = spark.table(f"{CATALOG}.silver.silver_payments") \
    .filter(F.col("payment_id").isNull())

In [0]:
def prepare_quarantine(df, source_name):
    return df.withColumn("record_id", F.lit(None)) \
        .withColumn("source_table", F.lit(source_name)) \
        .withColumn("error_reason", F.lit("validation_failed")) \
        .withColumn("error_severity", F.lit("HIGH")) \
        .withColumn("quarantine_timestamp", F.current_timestamp()) \
        .withColumn("source_file_name", F.lit("unknown")) \
        .withColumn("ingest_run_id", F.lit("run_001")) \
        .withColumn("original_record_json", F.to_json(F.struct(*df.columns)))

In [0]:
customers_q = prepare_quarantine(customers_invalid, "silver_customers")

write_quarantine(customers_q, "quarantine_invalid_customers")

In [0]:
policies_q = prepare_quarantine(policies_invalid, "silver_policies")

write_quarantine(policies_q, "quarantine_invalid_policies")

In [0]:
claims_q = prepare_quarantine(claims_invalid, "silver_claims")

write_quarantine(claims_q, "quarantine_invalid_claims")

In [0]:
payments_q = prepare_quarantine(payments_invalid, "silver_payments")

write_quarantine(payments_q, "quarantine_invalid_payments")

In [0]:
print("Customers:", spark.table(f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_customers").count())
print("Policies:", spark.table(f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_policies").count())
print("Claims:", spark.table(f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_claims").count())
print("Payments:", spark.table(f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_payments").count())